# PASO 2: Preprocesamiento de Datos
## Predicción de Deserción Estudiantil

**Objetivo:** Limpiar, transformar y preparar los datos para el modelado ML

In [1]:
# Importar librerías
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

✅ Librerías importadas


### 2.1 Cargar el Dataset

In [2]:
# Cargar dataset desde PASO 1
try:
    # Primero intenta desde archivo local (si lo guardaste en PASO 1)
    df = pd.read_csv('../data/processed/dataset_completo.csv')
    print(f"✅ Dataset cargado desde archivo local: {df.shape}")
except:
    # Si no existe, descárgalo de UCI
    print("📥 Descargando dataset de UCI...")
    url = "https://archive.ics.uci.edu/static/public/697/data.csv"
    df = pd.read_csv(url)
    print(f"✅ Dataset descargado: {df.shape}")
    # Guardar para futura referencia
    df.to_csv('../data/processed/dataset_completo.csv', index=False)

✅ Dataset cargado desde archivo local: (4424, 37)


In [3]:
# Exploración rápida
print(f"Dimensiones: {df.shape}")
print(f"\nColumnas: {df.columns.tolist()}")
print(f"\nTarget (últimas 5 valores): {df['Target'].unique()}")

Dimensiones: (4424, 37)

Columnas: ['Marital Status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Previous qualification (grade)', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Admission grade', 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'International', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'Unemployment rate', 'Inflation ra

### 2.2 Binarizar Target: Dropout=1, Resto=0

In [4]:
print("📊 Distribución original del Target:")
print(df['Target'].value_counts())
print(f"\nClases: {df['Target'].unique()}")

# Binarizar: Dropout=1, resto=0
df['Target_Binario'] = (df['Target'] == 'Dropout').astype(int)

print(f"\n✅ Target binarizado:")
print(df['Target_Binario'].value_counts())
print(f"\n  Dropout (1): {(df['Target_Binario'] == 1).sum()} casos")
print(f"  No Dropout (0): {(df['Target_Binario'] == 0).sum()} casos")

📊 Distribución original del Target:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

Clases: <StringArray>
['Dropout', 'Graduate', 'Enrolled']
Length: 3, dtype: str

✅ Target binarizado:
Target_Binario
0    3003
1    1421
Name: count, dtype: int64

  Dropout (1): 1421 casos
  No Dropout (0): 3003 casos


In [5]:
# Usar el target binarizado
y = df['Target_Binario'].copy()

# Eliminar columnas innecesarias
df_features = df.drop(columns=['Target', 'Target_Binario'])

print(f"✅ Features preparadas: {df_features.shape[1]} características")
print(f"✅ Target preparado: {y.shape[0]} muestras")

✅ Features preparadas: 36 características
✅ Target preparado: 4424 muestras


### 2.3 Imputación de Valores Nulos

In [6]:
# Verificar valores nulos
nulos = df_features.isnull().sum()
print(f"Valores nulos por columna:")
if nulos.sum() == 0:
    print("✅ No hay valores nulos")
else:
    print(nulos[nulos > 0])
    
    # Imputar nulos: media para numéricas, moda para categóricas
    numeric_cols = df_features.select_dtypes(include=['float64', 'int64']).columns
    categorical_cols = df_features.select_dtypes(include=['object']).columns
    
    # Media para numéricas
    for col in numeric_cols:
        if df_features[col].isnull().sum() > 0:
            df_features[col].fillna(df_features[col].mean(), inplace=True)
    
    # Moda para categóricas
    for col in categorical_cols:
        if df_features[col].isnull().sum() > 0:
            df_features[col].fillna(df_features[col].mode()[0], inplace=True)
    
    print("\n✅ Imputación completada")

Valores nulos por columna:
✅ No hay valores nulos


### 2.4 Codificación de Variables Categóricas

In [7]:
# Identificar columnas categóricas
categorical_cols = df_features.select_dtypes(include=['object']).columns.tolist()
print(f"Columnas categóricas detectadas: {len(categorical_cols)}")
print(f"Columnas: {categorical_cols}")

# Codificar con LabelEncoder
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_features[col] = le.fit_transform(df_features[col].astype(str))
    label_encoders[col] = le
    print(f"  ✅ {col}: {len(le.classes_)} clases codificadas")

print(f"\n✅ Todas las variables son numéricas ahora")

Columnas categóricas detectadas: 0
Columnas: []

✅ Todas las variables son numéricas ahora


### 2.5 Escalado de Features con StandardScaler

In [8]:
# Crear el escalador
scaler = StandardScaler()

# Aplicar escalado
X_scaled = scaler.fit_transform(df_features)
X_scaled = pd.DataFrame(X_scaled, columns=df_features.columns)

print(f"✅ Features escaladas con StandardScaler")
print(f"\nEstadísticas después del escalado:")
print(f"  Media: {X_scaled.mean().mean():.6f} (cercana a 0)")
print(f"  Desv. Est.: {X_scaled.std().mean():.6f} (cercana a 1)")
print(f"\nPrimeras 5 filas escaladas:")
print(X_scaled.head())

✅ Features escaladas con StandardScaler

Estadísticas después del escalado:
  Media: -0.000000 (cercana a 0)
  Desv. Est.: 1.000113 (cercana a 1)

Primeras 5 filas escaladas:
   Marital Status  Application mode  Application order    Course  \
0       -0.294829         -0.095470           2.490896 -4.209520   
1       -0.294829         -0.209869          -0.554068  0.192580   
2       -0.294829         -1.010660           2.490896  0.103404   
3       -0.294829         -0.095470           0.207173  0.444115   
4        1.356212          1.162916          -0.554068 -0.408389   

   Daytime/evening attendance  Previous qualification  \
0                    0.350082                -0.35023   
1                    0.350082                -0.35023   
2                    0.350082                -0.35023   
3                    0.350082                -0.35023   
4                   -2.856470                -0.35023   

   Previous qualification (grade)  Nacionality  Mother's qualification  \

### 2.6 División Train/Validation/Test (70/15/15)

In [9]:
# Paso 1: Separar train (70%) del resto (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y, 
    test_size=0.30, 
    random_state=42, 
    stratify=y  # Mantiene proporciones de clases
)

# Paso 2: Separar el 30% en validation (15%) y test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.50,  # 50% de 30% = 15%
    random_state=42, 
    stratify=y_temp
)

print("="*70)
print("DIVISIÓN TRAIN/VALIDATION/TEST")
print("="*70)
print(f"\n📊 Tamaños finales:")
print(f"  Train:      {X_train.shape[0]:5d} muestras ({X_train.shape[0]/len(X_scaled)*100:.1f}%)")
print(f"  Validation: {X_val.shape[0]:5d} muestras ({X_val.shape[0]/len(X_scaled)*100:.1f}%)")
print(f"  Test:       {X_test.shape[0]:5d} muestras ({X_test.shape[0]/len(X_scaled)*100:.1f}%)")
print(f"  Total:      {len(X_scaled):5d} muestras")

print(f"\n⚖️ Balance de clases (Dropout):")
print(f"  Train:      {y_train.sum()} Dropout ({y_train.mean()*100:.1f}%)")
print(f"  Validation: {y_val.sum()} Dropout ({y_val.mean()*100:.1f}%)")
print(f"  Test:       {y_test.sum()} Dropout ({y_test.mean()*100:.1f}%)")
print("\n✅ División completada con estratificación")

DIVISIÓN TRAIN/VALIDATION/TEST

📊 Tamaños finales:
  Train:       3096 muestras (70.0%)
  Validation:   664 muestras (15.0%)
  Test:         664 muestras (15.0%)
  Total:       4424 muestras

⚖️ Balance de clases (Dropout):
  Train:      994 Dropout (32.1%)
  Validation: 214 Dropout (32.2%)
  Test:       213 Dropout (32.1%)

✅ División completada con estratificación


### 2.7 Guardar Datos Procesados

In [10]:
# Guardar los splits como archivos pickle para el siguiente paso
import pickle

data_dict = {
    'X_train': X_train,
    'X_val': X_val,
    'X_test': X_test,
    'y_train': y_train,
    'y_val': y_val,
    'y_test': y_test,
    'scaler': scaler,
    'label_encoders': label_encoders,
    'feature_names': X_scaled.columns.tolist()
}

with open('../data/processed/preprocessed_data.pkl', 'wb') as f:
    pickle.dump(data_dict, f)

print("✅ Datos procesados guardados en: preprocessed_data.pkl")

# Guardar también como CSV para referencia
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_val.to_csv('../data/processed/y_val.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("✅ Sets de datos guardados como CSV")

✅ Datos procesados guardados en: preprocessed_data.pkl
✅ Sets de datos guardados como CSV


### 2.8 Resumen del Preprocesamiento

In [11]:
resumen = f"""
═══════════════════════════════════════════════════════════════════════════════
RESUMEN DEL PREPROCESAMIENTO
═══════════════════════════════════════════════════════════════════════════════

📊 DATASET ORIGINAL:
  • Muestras: {df.shape[0]:,}
  • Features: {df.shape[1]}

🎯 TARGET BINARIZADO:
  • Dropout (1): {(y == 1).sum():,} casos
  • No Dropout (0): {(y == 0).sum():,} casos
  • Proporción Dropout: {y.mean()*100:.2f}%

🔧 TRANSFORMACIONES APLICADAS:
  ✓ Binarización del Target (Dropout vs No Dropout)
  ✓ Manejo de valores nulos
  ✓ Codificación de {len(label_encoders)} variables categóricas
  ✓ Escalado StandardScaler (μ≈0, σ≈1)

📈 FEATURES FINALES: {X_train.shape[1]}
  • Todas numéricas
  • Escaladas entre -3 y +3 (aprox.)

✂️ DIVISIÓN DE DATOS (random_state=42, stratificado):
  
  TRAINING SET (70%):
    • Muestras: {X_train.shape[0]:,}
    • Dropout: {y_train.sum():,} ({y_train.mean()*100:.1f}%)
  
  VALIDATION SET (15%):
    • Muestras: {X_val.shape[0]:,}
    • Dropout: {y_val.sum():,} ({y_val.mean()*100:.1f}%)
  
  TEST SET (15%):
    • Muestras: {X_test.shape[0]:,}
    • Dropout: {y_test.sum():,} ({y_test.mean()*100:.1f}%)

📦 ARCHIVOS GENERADOS:
  ✓ preprocessed_data.pkl (diccionario con todos los datos + objetos)
  ✓ X_train.csv, X_val.csv, X_test.csv
  ✓ y_train.csv, y_val.csv, y_test.csv

✅ PREPROCESAMIENTO COMPLETADO
═══════════════════════════════════════════════════════════════════════════════
"""

print(resumen)

# Guardar resumen
with open('../reports/02_preprocessing_summary.txt', 'w', encoding='utf-8') as f:
    f.write(resumen)
print("\n✅ Resumen guardado en: 02_preprocessing_summary.txt")


═══════════════════════════════════════════════════════════════════════════════
RESUMEN DEL PREPROCESAMIENTO
═══════════════════════════════════════════════════════════════════════════════

📊 DATASET ORIGINAL:
  • Muestras: 4,424
  • Features: 38

🎯 TARGET BINARIZADO:
  • Dropout (1): 1,421 casos
  • No Dropout (0): 3,003 casos
  • Proporción Dropout: 32.12%

🔧 TRANSFORMACIONES APLICADAS:
  ✓ Binarización del Target (Dropout vs No Dropout)
  ✓ Manejo de valores nulos
  ✓ Codificación de 0 variables categóricas
  ✓ Escalado StandardScaler (μ≈0, σ≈1)

📈 FEATURES FINALES: 36
  • Todas numéricas
  • Escaladas entre -3 y +3 (aprox.)

✂️ DIVISIÓN DE DATOS (random_state=42, stratificado):

  TRAINING SET (70%):
    • Muestras: 3,096
    • Dropout: 994 (32.1%)

  VALIDATION SET (15%):
    • Muestras: 664
    • Dropout: 214 (32.2%)

  TEST SET (15%):
    • Muestras: 664
    • Dropout: 213 (32.1%)

📦 ARCHIVOS GENERADOS:
  ✓ preprocessed_data.pkl (diccionario con todos los datos + objetos)
  ✓ X

In [12]:
print("\n" + "="*70)
print("✅ PASO 2 COMPLETADO - Listo para PASO 3 (Modelado)")
print("="*70)


✅ PASO 2 COMPLETADO - Listo para PASO 3 (Modelado)
